# 面试题：有预算的规划搜索怎样设计？

回答要点：规划节点把累计成本、剩余预算、已验证事实和风险放进状态；展开动作前检查前置条件和预算，不允许低成本动作绕过政策。搜索使用可解释启发式选择候选，达到预算时返回已验证的部分结果。下面以旅行助手的只读信息收集任务，对比固定顺序和有预算 best-first 搜索。

## 真实案例

旅行助手要收集航班、酒店、差旅政策、天气、签证和行程摘要六类信息；每次工具调用都有虚拟成本和信息收益。

## 基线

基线按固定顺序调用，可能在预算耗尽前漏掉强制政策检查。

## 结果解读

手写搜索输出每次选中的动作、成本、剩余预算与覆盖的目标。

## 失败案例

只按收益排序会选天气而漏掉合规政策，修正为给强制目标硬门槛。

In [1]:
actions = [{'name':'查航班','cost':3,'gain':7,'target':'flight','required':True}, {'name':'查酒店','cost':3,'gain':6,'target':'hotel','required':True}, {'name':'查政策','cost':2,'gain':5,'target':'policy','required':True}, {'name':'查天气','cost':1,'gain':3,'target':'weather','required':False}, {'name':'查签证','cost':2,'gain':4,'target':'visa','required':False}, {'name':'生成摘要','cost':1,'gain':2,'target':'summary','required':False}]  # 构造六项只读工具及其虚拟成本、收益和强制属性。
budget = 8  # 设置用户允许的总工具调用预算。
print('动作输入:', [(item['name'], item['cost'], item['gain'], item['required']) for item in actions])  # 输出所有可选工具及属性。
print('总预算:', budget)  # 输出本教学任务的硬预算。

动作输入: [('查航班', 3, 7, True), ('查酒店', 3, 6, True), ('查政策', 2, 5, True), ('查天气', 1, 3, False), ('查签证', 2, 4, False), ('生成摘要', 1, 2, False)]
总预算: 8


In [2]:
fixed_order = ['查航班','查酒店','查天气','查签证','查政策','生成摘要']  # 定义忽略业务强制性的固定调用顺序。
fixed_cost = 0  # 初始化固定顺序已花费成本。
fixed_done = []  # 初始化固定顺序已完成动作。
for name in fixed_order:  # 按预设顺序尝试调用工具。
    item = next(action for action in actions if action['name'] == name)  # 读取当前动作的成本和收益。
    if fixed_cost + item['cost'] <= budget:  # 仅按剩余预算判断是否调用。
        fixed_cost += item['cost']  # 累加固定顺序的真实花费。
        fixed_done.append(name)  # 记录基线实际执行的动作。
print('固定顺序:', fixed_done, '，花费:', fixed_cost)  # 输出基线漏掉政策的执行结果。

固定顺序: ['查航班', '查酒店', '查天气', '生成摘要'] ，花费: 8


In [3]:
def search_with_budget(candidates, limit):  # 定义有预算且先满足强制目标的手写搜索器。
    remaining = list(candidates)  # 复制还可选择的候选动作。
    selected = []  # 初始化已经选择的动作列表。
    spent = 0  # 初始化累计成本。
    while remaining:  # 在仍有候选时继续扩展搜索状态。
        required_left = [item for item in remaining if item['required']]  # 找出尚未覆盖的强制目标。
        pool = required_left if required_left else remaining  # 强制目标存在时只在强制候选中搜索。
        best = max(pool, key=lambda item:item['gain'] / item['cost'])  # 以单位成本信息收益作为透明启发式。
        if spent + best['cost'] > limit:  # 检查当前候选是否超出硬预算。
            remaining.remove(best)  # 移除无法负担的候选并继续寻找可行项。
            continue  # 回到循环处理其他候选。
        selected.append(best)  # 将满足预算的最佳候选写入计划。
        spent += best['cost']  # 累计实际工具费用。
        remaining.remove(best)  # 防止同一只读动作重复进入计划。
    return selected, spent  # 返回可执行计划和总花费。

In [4]:
selected, spent = search_with_budget(actions, budget)  # 在相同预算下运行有约束的规划搜索。
print('动作 | 单价 | 收益 | 累计花费')  # 输出搜索轨迹表标题。
running = 0  # 初始化用于展示的累计费用。
for item in selected:  # 遍历按启发式选择的动作。
    running += item['cost']  # 更新当前搜索路径上的累计花费。
    print(item['name'], item['cost'], item['gain'], running)  # 输出每一步的可读中间量。
print('强制目标完成:', {item['target'] for item in selected if item['required']}, '，总花费:', spent)  # 输出预算下的合规覆盖与成本。

动作 | 单价 | 收益 | 累计花费
查政策 2 5 2
查航班 3 7 5
查酒店 3 6 8
强制目标完成: {'policy', 'flight', 'hotel'} ，总花费: 8


In [5]:
naive = sorted(actions, key=lambda item:item['gain'] / item['cost'], reverse=True)  # 构造没有强制门槛的纯收益排序。
naive_names = [item['name'] for item in naive[:3]]  # 读取预算近似下优先选中的前三项。
required_names = [item['name'] for item in selected if item['required']]  # 读取修正搜索覆盖的强制动作。
print('失败案例：纯收益前三=', naive_names, '，修正后的强制动作=', required_names)  # 展示合规不应被启发式收益压过。
print('生产差距：真实搜索需接入动态 token/时延预算、工具健康度、权限、缓存与搜索状态去重；本例不执行任何写操作。')  # 说明教学搜索边界。

失败案例：纯收益前三= ['查天气', '查政策', '查航班'] ，修正后的强制动作= ['查政策', '查航班', '查酒店']
生产差距：真实搜索需接入动态 token/时延预算、工具健康度、权限、缓存与搜索状态去重；本例不执行任何写操作。


In [6]:
assert spent <= budget  # 验证规划器从不超过用户硬预算。
assert '查政策' in required_names  # 验证强制政策检查不会被跳过。
assert '查天气' in naive_names  # 验证纯收益规则确实会错误偏向低成本天气。